# TARST 自定义唤醒词训练

本 Notebook 在 Google Colab/Linux GPU 上使用 openWakeWord 官方自动流程训练 `TARST.onnx`。

先运行 `TARST`，确认发音和误唤醒表现后，再运行 `Hey-TARST`。

In [ ]:
# 运行环境与工作目录
%cd /content
!mkdir -p tarst-training
%cd /content/tarst-training
!nvidia-smi || true

In [ ]:
# 直接在 Colab 运行时写入训练配置，不需要上传本机文件
from pathlib import Path

tarst_config = r'''model_name: "TARST"
target_phrase:
  - "TARST"
custom_negative_phrases:
  - "toast"
  - "trust"
  - "start"
  - "artist"
  - "tar"
n_samples: 20000
n_samples_val: 2000
tts_batch_size: 50
augmentation_batch_size: 16
piper_sample_generator_path: "./piper-sample-generator"
output_dir: "./TARST_model"
rir_paths:
  - "./mit_rirs"
background_paths:
  - "./background_clips"
background_paths_duplication_rate:
  - 1
false_positive_validation_data_path: "./validation_set_features.npy"
augmentation_rounds: 1
# 开发模式先不下载超大的 ACAV100M 特征；正式训练时再加入通用负样本
feature_data_files: {}
batch_n_per_class:
  adversarial_negative: 128
  positive: 128
model_type: "dnn"
layer_size: 32
steps: 50000
max_negative_weight: 1500
target_false_positives_per_hour: 0.2
'''
Path('TARST.yaml').write_text(tarst_config)
print('已写入 /content/tarst-training/TARST.yaml')

In [ ]:
# 安装训练依赖：使用独立 Python 3.11 环境，避免 Colab Python 3.12 与旧 Piper 冲突
!apt-get -qq update
!apt-get -qq install -y python3.11 python3.11-venv ffmpeg libsndfile1
!test -d openwakeword || git clone https://github.com/dscripka/openWakeWord.git openwakeword
!test -f piper-sample-generator/generate_samples.py || (rm -rf piper-sample-generator && git clone --depth 1 --branch v2.0.0 https://github.com/rhasspy/piper-sample-generator.git piper-sample-generator)
!test -f piper-sample-generator/models/en_US-libritts_r-medium.pt || wget -q -O piper-sample-generator/models/en_US-libritts_r-medium.pt https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt
# PyTorch 2.6+ 默认只加载权重；官方 Piper .pt 是可信的完整 checkpoint，需要允许完整反序列化
!{PY} -c "from pathlib import Path; p=Path('piper-sample-generator/generate_samples.py'); s=p.read_text(); s=s.replace('torch.load(model_path)', 'torch.load(model_path, weights_only=False)'); p.write_text(s)"
!test -d /content/tarst-venv || /usr/bin/python3.11 -m venv /content/tarst-venv
PY = '/content/tarst-venv/bin/python'
!{PY} -m pip install -q -U pip 'setuptools<81' wheel
!{PY} -m pip install -q 'numpy<2' 'scipy==1.14.1' 'torch==2.5.1' 'torchaudio==2.5.1'
!{PY} -m pip install -q piper-phonemize==1.1.0 webrtcvad-wheels mutagen==1.47.0 torchinfo==1.8.0 torchmetrics==1.2.0 speechbrain==0.5.14 audiomentations==0.33.0 torch-audiomentations==0.11.0 acoustics==0.2.6 pronouncing==0.2.0 deep-phonemizer==0.0.19
!{PY} -m pip install -q 'datasets>=2.14,<3' 'pyarrow<15' 'fsspec<2024.1.0'
# torch-audiomentations 旧版会调用已移除的 torchaudio.set_audio_backend；做兼容性补丁
!{PY} -c "from pathlib import Path; p=Path('/content/tarst-venv/lib/python3.11/site-packages/torch_audiomentations/utils/io.py'); s=p.read_text(); s=s.replace('torchaudio.set_audio_backend(\"soundfile\")', 'if hasattr(torchaudio, \"set_audio_backend\"): torchaudio.set_audio_backend(\"soundfile\")'); p.write_text(s)"
!{PY} -m pip install -q -e ./openwakeword
!{PY} -c 'import torch, numpy, pyarrow, datasets; print("python 3.11 environment OK"); print(torch.__version__, numpy.__version__, pyarrow.__version__, datasets.__version__)'

In [ ]:
# 下载 openWakeWord 的共享特征、验证集和训练所需的共享模型
!mkdir -p openwakeword/openwakeword/resources/models
!wget --show-progress -O openwakeword/openwakeword/resources/models/embedding_model.onnx https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx
!wget --show-progress -O openwakeword/openwakeword/resources/models/melspectrogram.onnx https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx
!wget --show-progress -O validation_set_features.npy https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy
# 生成可运行的开发级背景音频和简单房间响应；正式训练时应换成有授权的真实数据
import numpy as np
from scipy.io import wavfile
from pathlib import Path
rng = np.random.default_rng(42)
Path('mit_rirs').mkdir(exist_ok=True)
Path('background_clips').mkdir(exist_ok=True)
rir = np.zeros(16000, dtype=np.float32); rir[0] = 1.0; rir[2400] = 0.25; rir[5200] = 0.10
wavfile.write('mit_rirs/dev_room.wav', 16000, (rir * 32767).astype(np.int16))
for i in range(12):
    noise = rng.normal(0, 0.035 + i * 0.002, 16000 * 10).astype(np.float32)
    wavfile.write(f'background_clips/dev_noise_{i:02d}.wav', 16000, (np.clip(noise, -1, 1) * 32767).astype(np.int16))
print('共享模型、验证特征和开发级音频数据已准备好。')

## 准备背景音频

请将有明确授权的 16 kHz WAV 语音、噪声或音乐放入 `background_clips/`，并将房间脉冲响应 WAV 放入 `mit_rirs/`。可以直接运行官方 Notebook 中对应的数据下载单元。

首次运行建议先准备少量数据验证流程；正式模型再扩大数据规模。

In [ ]:
# 试听检查：先只生成少量 TARST 样本
# 正式运行前，请确认 Piper 对 TARST 的发音正确。
!TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 MPLBACKEND=Agg {PY} -u openwakeword/openwakeword/train.py --training_config TARST.yaml --generate_clips

In [ ]:
# 数据增强
!TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 MPLBACKEND=Agg {PY} -u openwakeword/openwakeword/train.py --training_config TARST.yaml --augment_clips

In [ ]:
# 训练并导出 ONNX/TFLite
!TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 MPLBACKEND=Agg {PY} -u openwakeword/openwakeword/train.py --training_config TARST.yaml --train_model
!ls -lh TARST_model/

训练成功后，下载 `TARST_model/TARST.onnx`。在 Colab 文件栏右键该文件选择 Download，然后在 TARST macOS 设置中导入。

训练 `Hey-TARST` 时，将后三个训练单元中的 `TARST.yaml` 替换为 `Hey-TARST.yaml`。